# Qrickstart

## 학습 내용

- FashionMNIST 데이터셋 사용
- DataLoader를 이용한 배치 처리
- nn.Module을 이용한 모델 정의
- Loss Function과 Optimizer 사용
- Train/Test Loop 구현
- GPU(CUDA) 사용
- 모델 저장 및 불러오기
- 단일 이미지 예측

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [ ]:
# 학습 데이터 다운로드
training_data = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)
# 테스트 데이터 다운로드
test_data = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
1
NVIDIA GeForce RTX 4060


In [ ]:
batch_size = 64

# Create data loaders
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

device = "cuda" if torch.cuda.is_available() else "cpu"

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")


    print("이동 전:", X.device)

    X = X.to(device)
    y = y.to(device)

    print("이동 후:", X.device)
    
    break

print("훈련 데이터:", len(training_data))
print("테스트 데이터:", len(test_data))

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전: cpu
이동 후: cuda:0
훈련 데이터: 60000
테스트 데이터: 10000


## 모델 생성

In [ ]:
# Define model

# 딥러닝 모델
class NeuralNetwork(nn.Module):
    def __init__(self):
        # 부모 클래스 (nn.Module) 초기화
        super().__init__()
        # 1차원 벡터로 펼침
        self.flatten = nn.Flatten()
        # 레이어를 순서대로 쌓는 방식
        self.linear_relu_stack = nn.Sequential(
            # 픽셀 784개를 받아서 feature 512개 생성
            nn.Linear(28*28, 512),
            # ReLU() 활성화 함수 음수면 제거, 양수면 그대로 출력
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            # 10개의 클래스에 대한 점수(logit) 출력
            nn.Linear(512, 10)
        )
    # 모델이 계산하는 흐름
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# 모델 자체를 GPU로 이동
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## 모델 매개변수 최적화

In [ ]:
# 손실 함수
loss_fn = nn.CrossEntropyLoss()
# 최적화 도구
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        # 예측값, 실제 정답 차이 계산
        loss = loss_fn(pred, y)

        # 역전파(Backpropagation)
        # 어떤 weight를 얼마나 수정해야 되는지 gradient 계산
        loss.backward()
        # 가중치 업데이트
        optimizer.step()
        # gradient 초기화
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

## 모델 성능 측정

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    # 평가 모드로 변경
    model.eval()
    # loss 누적, 맞춘 개수 누적
    test_loss, correct = 0, 0
    # gradient 계산 안 함
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            # batch 별 loss 더함
            test_loss += loss_fn(pred, y).item()
            # 가장 높은 점수의 클래스 index 선택 후 정답과 비교
            # True는 1.0 False는 0.0으로 변환 후 맞춘 개수 계산
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    
    # 전체 평균 loss 계산
    test_loss /= num_batches
    # accuracy
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, AVG loss: {test_loss:>8f} \n")

## Epochs

In [ ]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n----------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

print("Done")

Epoch 1
----------------------------
loss: 2.299336 [   64/60000]
loss: 2.285098 [ 6464/60000]
loss: 2.264567 [12864/60000]
loss: 2.259480 [19264/60000]
loss: 2.245697 [25664/60000]
loss: 2.210847 [32064/60000]
loss: 2.230494 [38464/60000]
loss: 2.188788 [44864/60000]
loss: 2.185842 [51264/60000]
loss: 2.160575 [57664/60000]
Test Error: 
 Accuracy: 38.4%, AVG loss: 2.145881 

Epoch 2
----------------------------
loss: 2.162312 [   64/60000]
loss: 2.145345 [ 6464/60000]
loss: 2.084056 [12864/60000]
loss: 2.098325 [19264/60000]
loss: 2.044997 [25664/60000]
loss: 1.988842 [32064/60000]
loss: 2.026350 [38464/60000]
loss: 1.938374 [44864/60000]
loss: 1.950590 [51264/60000]
loss: 1.878443 [57664/60000]
Test Error: 
 Accuracy: 52.5%, AVG loss: 1.867890 

Epoch 3
----------------------------
loss: 1.913742 [   64/60000]
loss: 1.871580 [ 6464/60000]
loss: 1.753321 [12864/60000]
loss: 1.791068 [19264/60000]
loss: 1.684621 [25664/60000]
loss: 1.647830 [32064/60000]
loss: 1.680866 [38464/60000]
lo

## 모델 저장

In [ ]:
torch.save(model.state_dict(), "../models/fashion_mnist_model.pth")
print("Save PyTorch Model State to model.pth")

Save PyTorch Model State to model.pth


In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("../models/fashion_mnist_model.pth", weights_only=True))

<All keys matched successfully>

## Predition

In [ ]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

model.eval()
# x = 이미지, y = 정답 label 번호
x, y = test_data[0]
# 학습 모드 off
with torch.no_grad():
    x = x.to(device)
    # batch 차원 추가
    x = x.unsqueeze(0)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'"Predicted: "{predicted}", Actual: "{actual}"')

"Predicted: "Ankle boot", Actual: "Ankle boot"


## 정리

- PyTorch 학습의 전체 흐름을 이해한다.
- Dataset -> DataLoader -> Model -> Loss -> Optimizer -> Train/Test 과정 실습
- FashionMNIST를 이용하여 간단한 분류 모델을 학습했다.